# Group2 Baseline Workflow (Orchestration)

This notebook keeps heavy logic in `src/group2_stage2` and only orchestrates steps.


## Stage 0 Checklist
- [ ] Group1 environment (`.venv`) is active
- [ ] `configs/workflow_paths_subset_10000.json` is updated
- [ ] Stage2 variant datasets exist under `stage2_root/<variant>/stage2_dataset.jsonl`


In [ ]:
import json
import sys
from pathlib import Path

# Resolve project root robustly even if notebook cwd changes.
project_root = Path.cwd().resolve()
if (project_root / "configs" / "workflow_paths_subset_10000.json").exists():
    pass
elif (project_root.parent / "configs" / "workflow_paths_subset_10000.json").exists():
    project_root = project_root.parent
else:
    # Fallback for unusual launch locations.
    project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / "configs" / "workflow_paths_subset_10000.json"
raw_cfg = json.loads(config_path.read_text(encoding="utf-8"))

cfg = {
    k: (v.replace("${PROJECT_ROOT}", str(project_root)) if isinstance(v, str) else v)
    for k, v in raw_cfg.items()
}

stage2_root = Path(cfg["stage2_root"])
image_root = Path(cfg["image_root"])
feature_root = Path(cfg["clip_feature_root"])
reuse_feature_roots = [Path(p) for p in cfg.get("reuse_clip_feature_roots", [])]
quality_variants = cfg.get("quality_variants", [])
if not isinstance(quality_variants, list):
    quality_variants = []
all_variants = [str(cfg["baseline_variant"])] + [str(v) for v in quality_variants]

print("project_root:", project_root)
print("stage2_root:", stage2_root)
print("all_variants:", all_variants)
print("feature_root (write):", feature_root)
print("reuse_feature_roots:", [str(p) for p in reuse_feature_roots])


## Stage 1: Audit + Shared Split


In [ ]:
from src.group2_stage2.data.audit import audit_stage2_variants
from src.group2_stage2.data.splits import build_shared_quality_pool, materialize_train_val_split

audit = audit_stage2_variants(stage2_root, all_variants)
audit


In [ ]:
pool = build_shared_quality_pool(
    stage2_root=stage2_root,
    all_variants=all_variants,
    quality_image_count=int(cfg["quality_image_count"]),
    val_image_count=int(cfg["val_image_count"]),
    split_seed=int(cfg.get("split_seed", 42)),
    pool_reference_variant=cfg["baseline_variant"],
)
pool


In [ ]:
split_stats = materialize_train_val_split(stage2_root, all_variants)
split_stats


## Stage 2 Checklist (model-dependent)
- [ ] Group1 CLIP bundle loaded
- [ ] Group1 tokenizer loaded
- [ ] `get_features_compiled` ready from Group1 CLIP helper


In [ ]:
from src.group2_stage2.bootstrap_runtime import create_stage2_runtime_objects

clip_bundle = globals().get("clip_bundle")
get_features_compiled = globals().get("get_features_compiled")
tokenizer = globals().get("tokenizer")

if clip_bundle is None or get_features_compiled is None or tokenizer is None:
    runtime = create_stage2_runtime_objects(cfg)
    clip_bundle = runtime["clip_bundle"]
    get_features_compiled = runtime["get_features_compiled"]
    tokenizer = runtime["tokenizer"]
    print("runtime initialized (missing objects were built).")
else:
    print("runtime reuse: using existing tokenizer/clip_bundle/get_features_compiled from current session")

print("clip hidden_size:", getattr(clip_bundle, "hidden_size", "<unknown>"))
print("tokenizer loaded:", type(tokenizer).__name__)


In [ ]:
from src.group2_stage2.data.pipeline import prepare_stage2_variant_splits

required = ["tokenizer", "clip_bundle", "get_features_compiled"]
missing = [name for name in required if name not in globals()]

if missing:
    print("Stage 2 prerequisites missing:", missing)
    print("Initialize these first (typically from Group1 runtime):")
    print("- tokenizer")
    print("- clip_bundle")
    print("- get_features_compiled")
else:
    OVERWRITE_STAGE2_PREP = False
    STAGE2_VARIANTS_TO_PREP = [all_variants[0]]  # safe default: baseline only
    STAGE2_SPLITS_TO_PREP = ("val",)          # safe default: smaller prep first

    prep_results = prepare_stage2_variant_splits(
        stage2_root=stage2_root,
        image_root=image_root,
        feature_root=feature_root,
        tokenizer=tokenizer,
        clip_bundle=clip_bundle,
        get_features_compiled=get_features_compiled,
        all_variants=STAGE2_VARIANTS_TO_PREP,
        splits=STAGE2_SPLITS_TO_PREP,
        overwrite=OVERWRITE_STAGE2_PREP,
        additional_feature_roots=reuse_feature_roots,
    )
    print("Prepared variant/split jobs:", len(prep_results))
    print(prep_results[0] if prep_results else "<no results>")


## Stage 3: Evaluation Artifacts

### Checklist - Before Stage 3
- [ ] Stage 1 split files exist (`shared_quality_pool.json`, `shared_split.json`)
- [ ] Stage 2 variant datasets exist (`<variant>/stage2_dataset.jsonl`)
- [ ] You want diagnostics/sampling outputs under `stage2_instruction/`


In [ ]:
from src.group2_stage2.eval.quality_eval import (
    build_dataset_quality_diagnostics,
    build_qualitative_samples_pack,
    build_pairwise_judge_requests,
)

quality = build_dataset_quality_diagnostics(stage2_root, all_variants)
qual_samples = build_qualitative_samples_pack(stage2_root, all_variants)

# Requires heldout_eval_pack.json already prepared:
# pairwise = build_pairwise_judge_requests(stage2_root, baseline_variant=cfg["baseline_variant"])


## Stage 4: Quality Experiment Tracking

### Checklist - Before Stage 4
- [ ] Stage 2 train/val manifests exist for each variant
- [ ] `run_stage2_experiment` function is available (if you want to run model experiments here)
- [ ] `all_results_manual.json` is expected to contain one result per variant


In [ ]:
from src.group2_stage2.experiments.experiment_tracking import (
    select_next_variant,
    run_and_store_variant,
    prompt_alignment_audit,
    build_engine_comparison_summary,
    build_baseline_relative_comparison,
)

results_path = stage2_root / "all_results_manual.json"
selection = select_next_variant(
    results_path=results_path,
    expected_variants=all_variants,
    current_variant=None,
    allow_overwrite=False,
)
selection


In [ ]:
# Stage 4 run orchestration
RUN_STAGE4_EXPERIMENT = False
ALLOW_STAGE4_OVERWRITE = False
OVERWRITE_STAGE4_AUDIT = False

if "selection" not in globals():
    selection = select_next_variant(
        results_path=results_path,
        expected_variants=all_variants,
        current_variant=None,
        allow_overwrite=ALLOW_STAGE4_OVERWRITE,
    )

runner = globals().get("run_stage2_experiment")
if selection["selected_variant"] is None:
    print("All expected variants already exist in", results_path)
elif not RUN_STAGE4_EXPERIMENT:
    print("Selected next variant:", selection["selected_variant"])
    print("Set RUN_STAGE4_EXPERIMENT=True to execute and store it.")
elif runner is None:
    print("Stage 4 blocked: run_stage2_experiment is not defined in this session.")
else:
    run_result = run_and_store_variant(
        results_path=results_path,
        all_results=selection["all_results"],
        selected_variant=selection["selected_variant"],
        run_stage2_experiment_fn=runner,
    )
    print("Stored variant result:", run_result["ran_variant"])

audit_result = prompt_alignment_audit(
    stage2_root,
    all_variants,
    prompt_reference_variant=all_variants[0],
    overwrite=OVERWRITE_STAGE4_AUDIT,
)
print("Prompt alignment audit:", audit_result.get("mode", "generated"))


In [ ]:
import json

OVERWRITE_STAGE4_SUMMARY = False
if not results_path.exists():
    print("Stage 4 blocked: missing", results_path)
else:
    all_results = json.loads(results_path.read_text(encoding="utf-8"))
    missing = [v for v in all_variants if v not in all_results]
    if missing:
        print("Stage 4 summary blocked: missing variants in results file:", missing)
        print("Run/store experiments for missing variants first.")
    else:
        engine_summary = build_engine_comparison_summary(
            stage2_root,
            results_path,
            expected_variants=all_variants,
            overwrite=OVERWRITE_STAGE4_SUMMARY,
        )
        baseline_relative = build_baseline_relative_comparison(
            stage2_root,
            results_path,
            baseline_variant=cfg["baseline_variant"],
            expected_variants=all_variants,
            overwrite=OVERWRITE_STAGE4_SUMMARY,
        )
        print("Engine summary:", engine_summary.get("mode", "generated"))
        print("Baseline-relative summary:", baseline_relative.get("mode", "generated"))
        print("Best variant:", engine_summary.get("best_variant"))


## Stage 5: Quantity Ablation

### Checklist - Before Stage 5
- [ ] `engine_comparison_summary.json` exists (from Stage 4 summary)
- [ ] Shared split files exist (`shared_quality_pool.json`, `shared_split.json`)
- [ ] You understand quantity variants reuse one source variant chosen from Stage 4 ranking


In [ ]:
from src.group2_stage2.experiments.quantity_ablation import (
    derive_quantity_plan,
    build_quantity_variants,
    register_quantity_variants,
    prepare_quantity_variants,
    run_quantity_experiments,
    summarize_quantity_results,
)

engine_summary = stage2_root / "engine_comparison_summary.json"
if not engine_summary.exists():
    print("Stage 5 blocked: missing", engine_summary)
    print("Run Stage 4 engine summary first.")
else:
    plan = derive_quantity_plan(stage2_root)
    print("Quantity source variant:", plan["quantity_source_variant"])
    print("Quantity levels:", plan["quantity_levels"])
    print("Quantity split seed:", plan["quantity_split_seed"])


In [ ]:
OVERWRITE_STAGE5_VARIANTS = False

if "plan" not in globals():
    engine_summary = stage2_root / "engine_comparison_summary.json"
    if not engine_summary.exists():
        print("Stage 5 blocked: plan is missing and engine summary is not available.")
        print("Run Stage 4 summary first.")
    else:
        plan = derive_quantity_plan(stage2_root)

if "plan" in globals():
    quantity_variants = build_quantity_variants(
        stage2_root=stage2_root,
        quantity_source_variant=plan["quantity_source_variant"],
        quantity_levels=plan["quantity_levels"],
        quantity_split_seed=plan["quantity_split_seed"],
        overwrite=OVERWRITE_STAGE5_VARIANTS,
    )
    registration = register_quantity_variants(
        stage2_root,
        quantity_variants,
        overwrite=OVERWRITE_STAGE5_VARIANTS,
    )
    print("Quantity variants:", quantity_variants)
    print("Registered variants:", len(registration))


In [ ]:
from src.group2_stage2.data.tokenization import tokenize_stage2_variant
from src.group2_stage2.data.features import extract_stage2_features
from src.group2_stage2.data.manifests import build_stage2_manifest

PREP_QUANTITY_INPUTS = False
RUN_QUANTITY_EXPERIMENTS = False
OVERWRITE_STAGE5_PREP = False
ALLOW_STAGE5_RESULT_OVERWRITE = False

if "quantity_variants" not in globals() or not quantity_variants:
    print("Stage 5 blocked: quantity_variants is missing. Run previous Stage 5 cell first.")
else:
    if PREP_QUANTITY_INPUTS:
        required = ["tokenizer", "clip_bundle", "get_features_compiled"]
        missing = [name for name in required if name not in globals()]
        if missing:
            print("Stage 5 prep blocked: missing runtime objects:", missing)
        else:
            def tokenize_cb(variant, split, overwrite=False):
                return tokenize_stage2_variant(stage2_root, tokenizer, variant, split, overwrite=overwrite)

            def feature_cb(variant, split, overwrite=False):
                return extract_stage2_features(
                    stage2_root,
                    image_root,
                    feature_root,
                    clip_bundle,
                    get_features_compiled,
                    variant,
                    split,
                    overwrite=overwrite,
                )

            def manifest_cb(variant, split, overwrite=False):
                return build_stage2_manifest(stage2_root, feature_root, variant, split, overwrite=overwrite)

            prep_status = prepare_quantity_variants(
                stage2_root,
                quantity_variants,
                tokenize_fn=tokenize_cb,
                extract_features_fn=feature_cb,
                build_manifest_fn=manifest_cb,
                overwrite=OVERWRITE_STAGE5_PREP,
            )
            print("Quantity prep status entries:", len(prep_status))
    else:
        print("Set PREP_QUANTITY_INPUTS=True to build quantity tokenized/features/manifests.")

    runner = globals().get("run_stage2_experiment")
    if RUN_QUANTITY_EXPERIMENTS:
        if runner is None:
            print("Stage 5 run blocked: run_stage2_experiment is not defined.")
        else:
            quantity_results = run_quantity_experiments(
                stage2_root,
                quantity_variants,
                run_stage2_experiment_fn=runner,
                allow_overwrite=ALLOW_STAGE5_RESULT_OVERWRITE,
            )
            print("Quantity experiment results:", len(quantity_results))
    else:
        print("Set RUN_QUANTITY_EXPERIMENTS=True to execute quantity training runs.")

    if (stage2_root / "quantity_results.json").exists():
        summarize_quantity_results(stage2_root)
        print("Quantity summary written.")
    else:
        print("Quantity summary skipped: quantity_results.json not found yet.")


## Stage 6: Heldout Pack + Pairwise Requests

### Checklist - Before Stage 6
- [ ] Shared split exists (`shared_split.json`)
- [ ] Variant datasets exist for all compared models
- [ ] Choose baseline variant in config (`baseline_variant`)


In [ ]:
from src.group2_stage2.eval.evaluation_pack import build_heldout_eval_pack
from src.group2_stage2.eval.quality_eval import build_pairwise_judge_requests

heldout = build_heldout_eval_pack(stage2_root, all_variants, samples_per_task=10, seed=123)
# pairwise = build_pairwise_judge_requests(stage2_root, baseline_variant=cfg["baseline_variant"], seed=2026)
heldout["num_samples"]


## Stage 7: Reporting Figures

### Checklist - Before Stage 7
- [ ] `engine_comparison_summary.json` exists
- [ ] `baseline_relative_comparison.json` exists
- [ ] Optional: quantity summary exists if you ran Stage 5 experiments


In [ ]:
from src.group2_stage2.eval.reporting import build_engine_plots_and_table

engine_summary = stage2_root / "engine_comparison_summary.json"
quality_diag = stage2_root / "dataset_quality_diagnostics.json"
if not engine_summary.exists() or not quality_diag.exists():
    print("Stage 7 blocked: required files missing")
    print("-", engine_summary, "exists=", engine_summary.exists())
    print("-", quality_diag, "exists=", quality_diag.exists())
else:
    build_engine_plots_and_table(stage2_root)


## Appendix: Reset Snapshot Helper

Use this helper between variant runs to keep experiments fair.
It is not a separate pipeline stage.

### Checklist - Before Reset
- [ ] Stage-2 state objects exist in session (`projector_state`, `llama_state`, `opt_state`)
- [ ] Use one saved snapshot for fair repeated experiments

- Save a clean Stage-2 state snapshot once (after state objects exist).
- Restore that snapshot before each variant run for fair comparison.


In [ ]:
from src.group2_stage2.experiments.training_orchestration import save_stage2_snapshot

reset_root = stage2_root / "_fresh_state_snapshot"
required_state = ["projector_state", "llama_state", "opt_state"]
missing_state = [name for name in required_state if name not in globals()]

if missing_state:
    print("Cannot save snapshot. Missing state objects:", missing_state)
else:
    g = globals()
    snapshot_paths = save_stage2_snapshot(
        reset_root=reset_root,
        projector_state=g["projector_state"],
        llama_state=g["llama_state"],
        opt_state=g["opt_state"],
    )
    print("Saved Stage-2 snapshot:", snapshot_paths)


In [ ]:
from src.group2_stage2.experiments.training_orchestration import load_stage2_snapshot

reset_root = stage2_root / "_fresh_state_snapshot"
if not reset_root.exists():
    print("Snapshot folder not found:", reset_root)
    print("Run the save snapshot cell first.")
else:
    loaded = load_stage2_snapshot(reset_root)
    projector_state = loaded["projector_state"]
    llama_state = loaded["llama_state"]
    opt_state = loaded["opt_state"]
    print("Restored Stage-2 states from:", reset_root)
